[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_68_CLI_Polish_PyPI_Packaging.ipynb)

# Lesson 68 -- CLI Polish + PyPI Packaging for `agent-bench`

**Phase 7, Lesson 4 of ~6 (tentative).** Picks up exactly where L67 left off: `AsyncBenchmarkRunner` works, `registry.py` resolves environments/scorers by name, and L66 proved third-party plugins can be discovered via real `importlib.metadata.entry_points()`. What's still missing is everything L58 already built for `paper-distiller` back in Phase 6: a real, installable, `pip install`-able command-line tool with a version flag, a help screen, JSON output for scripting, and a PyPI release runbook. Today `agent-bench` gets the same treatment.

## Phase 7 Roadmap (tentative -- adapts as we go, same policy every phase has followed)

| Lesson | Topic | Status |
|---|---|---|
| L65 | Phase 7 Kickoff -- `agent-bench` as a second flagship OSS tool, plugin registry pattern | done |
| L66 | Real plugins -- `ShellEnv` + `importlib.metadata.entry_points()` | done |
| L67 | Performance -- `asyncio.gather` + `Semaphore`, async parallel execution | done |
| **L68** | **CLI polish + PyPI packaging (this lesson)** | **you are here** |
| L69 | OSS Growth -- README, badges, CONTRIBUTING funnel (following L60's playbook) | tentative |
| L70 | Launch Day -- Phase 7 capstone, actually ship `agent-bench` v0.1.0 | tentative |

**A loose end this lesson closes first:** L67's "condensed from L61/L65" `registry.py` quietly dropped L66's `discover_plugins()` / `load_plugins()` functions -- they weren't needed for the async-execution lesson, so they got left out of the recreation. That's a real instance of the exact risk L64 called out during `paper-distiller`'s integration test: modules validated in isolation can silently diverge from each other. Today's CLI needs plugin discovery (`list-plugins`), so step 1 is merging L66's plugin functions back into L67's class-based registry -- on purpose, not by accident this time.

## Concept: from "a notebook cell you can run" to "a tool you can install"

L58 drew this comparison for `paper-distiller`; the same table applies to `agent-bench` today.

| | Minimal demo CLI (L66/L67) | Production CLI (today) |
|---|---|---|
| Invocation | `typer.testing.CliRunner` only, inside the notebook | Real `agent-bench` console-script on `$PATH`, installed via pip |
| Commands | One collapsed command (`run`) or two ad hoc ones | `run` / `list-environments` / `list-agents` / `list-scorers` / `list-plugins`, all under one `Typer()` app |
| `--version` | Doesn't exist | Eager callback reading the **installed** distribution's version via `importlib.metadata`, not a hardcoded string |
| Output | `print()` / a Rich table, human-only | `--format table` or `json` -- json output is script/CI-safe |
| Distribution | Lives in the notebook's Python process only | `pyproject.toml` -> wheel + sdist -> `pip install agent-bench` from anywhere |
| Failure signal | An assertion in the notebook | Process exit code (`0` pass / `1` fail), the thing CI and shell scripts actually check |

Two things stay identical to L58's `paper-distiller` playbook: the `hatchling` build backend and the `[project.scripts]` entry-point mechanism. One thing is new and specific to `agent-bench`: `list-plugins` exposes L66's `entry_points()` discovery through the CLI, something `paper-distiller` never needed because it has no plugin system.

In [ ]:
# Setup
!pip install -q "typer>=0.9" "pydantic>=2.0" "rich>=13" hatchling build twine nest_asyncio 2>/dev/null

import os, sys, shutil, subprocess, textwrap, sysconfig, json as _json
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()  # Colab/Jupyter kernels already run an event loop; the CLI's own
                       # `asyncio.run()` (Step 2) would otherwise collide with it the moment
                       # we invoke `run` from inside THIS notebook -- irrelevant in a real
                       # terminal, where no loop is running yet. Pitfall #9 below, applied.

CONTENT = Path("/content")
CONTENT.mkdir(exist_ok=True)
PKG_DIR = CONTENT / "agent_bench_pkg"

def write_file(rel_path: str, text: str) -> Path:
    p = PKG_DIR / rel_path
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(textwrap.dedent(text).lstrip("\n"))
    return p

def run(*args, check=False):
    result = subprocess.run(list(args), capture_output=True, text=True)
    print("$ " + " ".join(args) + "  (exit=" + str(result.returncode) + ")")
    if result.returncode != 0:
        print(result.stdout[-1500:]); print(result.stderr[-1500:])
    if check and result.returncode != 0:
        raise RuntimeError("command failed: " + " ".join(args))
    return result

print("Ready. PKG_DIR:", PKG_DIR)

## Step 1 -- the package, merged and reconciled

`core.py` and `environments.py`/`runner_async.py` are recreated exactly as L67 left them (Task/Trajectory/TaskResult, class-based `ENVIRONMENT_REGISTRY`, `CalcEnv`/`FileEnv`/`MockAgent`, `AsyncBenchmarkRunner.run_sync`/`run_async`). `registry.py` is the one file that changes: L67's registration/lookup functions stay, and L66's `discover_plugins()`/`load_plugins()` come back in, adapted to the class-based registry.

In [ ]:
write_file("agent_bench/__init__.py", "")

write_file("agent_bench/core.py", """
# agent_bench/core.py -- Task/Trajectory/TaskResult data models + pass_at_k. Unchanged from L67.
from __future__ import annotations
import math
from dataclasses import dataclass, field
from typing import Any, Optional
from pydantic import BaseModel


class Task(BaseModel):
    id: str
    category: str
    difficulty: str = "medium"
    prompt: str
    env_name: str
    scorer_name: str


@dataclass
class TrajectoryStep:
    action: str
    result: Any


@dataclass
class Trajectory:
    task_id: str
    steps: list = field(default_factory=list)
    final_state: Any = None


@dataclass
class TaskResult:
    task_id: str
    attempt: int
    passed: bool
    trajectory: Optional[Trajectory]
    error: Optional[str]
    elapsed: float


def pass_at_k(n: int, c: int, k: int) -> float:
    # Unbiased pass@k estimator (Chen et al. 2021).
    if n - c < k:
        return 1.0
    return 1.0 - math.prod((n - c - i) / (n - i) for i in range(k))
""")
print("core.py written")

In [ ]:
write_file("agent_bench/registry.py", """
# agent_bench/registry.py -- registries (L67 class-based) + plugin discovery (L66, merged back today).
from __future__ import annotations
import sys
from importlib.metadata import entry_points
from typing import Any, Callable

ENVIRONMENT_REGISTRY = {}
AGENT_REGISTRY = {}
SCORER_REGISTRY = {}

_ENTRY_POINT_GROUPS = {
    "environments": "agent_bench.environments",
    "agents": "agent_bench.agents",
    "scorers": "agent_bench.scorers",
}


def register_environment(name: str):
    def deco(cls):
        if name in ENVIRONMENT_REGISTRY:
            raise ValueError("Environment already registered: " + name)
        ENVIRONMENT_REGISTRY[name] = cls
        return cls
    return deco


def register_agent(name: str):
    def deco(obj):
        if name in AGENT_REGISTRY:
            raise ValueError("Agent already registered: " + name)
        AGENT_REGISTRY[name] = obj
        return obj
    return deco


def register_scorer(name: str):
    def deco(fn):
        if name in SCORER_REGISTRY:
            raise ValueError("Scorer already registered: " + name)
        SCORER_REGISTRY[name] = fn
        return fn
    return deco


def get_environment(name: str):
    # Returns a FRESH instance every call -- the L67 fix, not an optimization.
    if name not in ENVIRONMENT_REGISTRY:
        raise KeyError("No environment registered as " + name + ". Known: " + str(sorted(ENVIRONMENT_REGISTRY)))
    return ENVIRONMENT_REGISTRY[name]()


def get_agent(name: str):
    if name not in AGENT_REGISTRY:
        raise KeyError("No agent registered as " + name + ". Known: " + str(sorted(AGENT_REGISTRY)))
    return AGENT_REGISTRY[name]


def get_scorer(name: str):
    if name not in SCORER_REGISTRY:
        raise KeyError("No scorer registered as " + name + ". Known: " + str(sorted(SCORER_REGISTRY)))
    return SCORER_REGISTRY[name]


def discover_plugins(kind: str = "environments"):
    # Read-only: list installed distributions advertising this entry-point group.
    group = _ENTRY_POINT_GROUPS[kind]
    eps = entry_points(group=group)
    return sorted(ep.name + " -> " + ep.value for ep in eps)


def load_plugins(kind: str = "environments", verbose: bool = True):
    # Discover AND import every installed plugin under this entry-point group.
    # Importing triggers @register_* decorators as an import side effect (L66).
    group = _ENTRY_POINT_GROUPS[kind]
    eps = entry_points(group=group)
    loaded = []
    for ep in eps:
        try:
            ep.load()
            loaded.append(ep.name)
            if verbose:
                dist_name = ep.dist.name if ep.dist else "?"
                print("  [plugin] loaded " + ep.name + " from " + ep.value + " (distribution: " + dist_name + ")")
        except Exception as e:
            print("  [plugin] FAILED to load " + ep.name + " (" + ep.value + "): " + str(e), file=sys.stderr)
    return loaded
""")
print("registry.py written -- L66 discover_plugins/load_plugins merged back into L67 class-based registry")

In [ ]:
write_file("agent_bench/environments.py", """
# agent_bench/environments.py -- CalcEnv/FileEnv/MockAgent/scorers, unchanged from L67.
from __future__ import annotations
import time
import random
from agent_bench.core import Task, Trajectory, TrajectoryStep
from agent_bench.registry import register_environment, register_scorer

LATENCY = 0.05  # seconds -- stand-in for one real Claude tool-call round trip


@register_environment("calc")
class CalcEnv:
    def __init__(self):
        self.history = []

    def step(self, expr: str):
        val = eval(expr, {"__builtins__": {}}, {})
        self.history.append((expr, val))
        return val

    def final_state(self):
        return self.history[-1][1] if self.history else None


@register_environment("file")
class FileEnv:
    def __init__(self):
        self.files = {}

    def write(self, path: str, content: str):
        self.files[path] = content

    def read(self, path: str):
        return self.files.get(path)

    def final_state(self):
        return dict(self.files)


class MockAgent:
    def __init__(self, flake_rate: float = 0.0, latency: float = LATENCY):
        self.flake_rate = flake_rate
        self.latency = latency

    def act(self, task, env):
        time.sleep(self.latency)
        if random.random() < self.flake_rate:
            raise RuntimeError("simulated transient API error")
        if task.env_name == "calc":
            expr = task.prompt.split("Compute: ")[-1].split(" = ")[0]
            val = env.step(expr)
            return Trajectory(task_id=task.id, steps=[TrajectoryStep("calc", val)], final_state=val)
        elif task.env_name == "file":
            fname = task.id + ".txt"
            env.write(fname, "result-for-" + task.id)
            return Trajectory(task_id=task.id, steps=[TrajectoryStep("write", fname)], final_state=env.final_state())
        return Trajectory(task_id=task.id, steps=[], final_state=None)


@register_scorer("exact_match")
def scorer_exact_match(trajectory, task):
    expected = float(task.prompt.split("=")[-1].strip()) if "=" in task.prompt else None
    return expected is not None and trajectory.final_state == expected


@register_scorer("file_written")
def scorer_file_written(trajectory, task):
    fname = task.id + ".txt"
    return isinstance(trajectory.final_state, dict) and fname in trajectory.final_state
""")

write_file("agent_bench/runner_async.py", """
# agent_bench/runner_async.py -- Semaphore-bounded concurrent execution + sequential baseline. Unchanged from L67.
from __future__ import annotations
import asyncio
import time
from agent_bench.core import Task, TaskResult
from agent_bench.registry import get_environment, get_scorer


class AsyncBenchmarkRunner:
    def __init__(self, tasks):
        self.tasks = tasks

    async def _run_one(self, agent, task, attempt, sem, loop):
        async with sem:
            env = get_environment(task.env_name)
            scorer = get_scorer(task.scorer_name)
            start = time.perf_counter()
            try:
                trajectory = await loop.run_in_executor(None, agent.act, task, env)
                passed = scorer(trajectory, task)
                return TaskResult(task.id, attempt, passed, trajectory, None, time.perf_counter() - start)
            except Exception as e:
                return TaskResult(task.id, attempt, False, None, str(e), time.perf_counter() - start)

    async def run_async(self, agent, k: int = 1, concurrency: int = 4):
        sem = asyncio.Semaphore(concurrency)
        loop = asyncio.get_event_loop()
        coros = [self._run_one(agent, task, a, sem, loop) for task in self.tasks for a in range(k)]
        return await asyncio.gather(*coros)

    def run_sync(self, agent, k: int = 1):
        results = []
        for task in self.tasks:
            for attempt in range(k):
                env = get_environment(task.env_name)
                scorer = get_scorer(task.scorer_name)
                start = time.perf_counter()
                try:
                    trajectory = agent.act(task, env)
                    passed = scorer(trajectory, task)
                    results.append(TaskResult(task.id, attempt, passed, trajectory, None, time.perf_counter() - start))
                except Exception as e:
                    results.append(TaskResult(task.id, attempt, False, None, str(e), time.perf_counter() - start))
        return results
""")
print("environments.py + runner_async.py written")

## Step 2 -- the CLI

Five commands under one `Typer()` app: `run`, `list-environments`, `list-agents`, `list-scorers`, `list-plugins`. A `--version` eager callback reads the version off the **installed distribution** via `importlib.metadata.version()` rather than a hardcoded constant -- so it can never drift from what `pip show agent-bench` reports, the same discipline L58 established for `paper-distiller`. `run` supports `--format table|json` so it can be piped into `jq` or a CI step, and exits `1` on any failed attempt (Typer's `raise typer.Exit(code=1)`) so shells and CI systems can branch on it without parsing text.

One asymmetry worth flagging honestly rather than papering over: `list-plugins` works for `environments` because L66 built a real installable `ShellEnv` plugin package for that entry-point group. `agents` and `scorers` have the same `_ENTRY_POINT_GROUPS` wiring in `registry.py`, but no third-party package has ever registered under those groups -- so `list-plugins --kind agents` will legitimately return empty today. That's flagged as homework, not hidden.

In [ ]:
write_file("agent_bench/cli.py", """
# agent_bench/cli.py -- production Typer CLI. New this lesson.
from __future__ import annotations
import asyncio
import json as _json
from importlib.metadata import version as _pkg_version, PackageNotFoundError

import typer
from rich.console import Console
from rich.table import Table

from agent_bench.core import Task
from agent_bench.registry import (
    ENVIRONMENT_REGISTRY, AGENT_REGISTRY, SCORER_REGISTRY,
    discover_plugins, load_plugins,
)
from agent_bench.environments import MockAgent  # built-ins self-register on import
from agent_bench.runner_async import AsyncBenchmarkRunner

app = typer.Typer(rich_markup_mode="rich", help="agent-bench: a pluggable benchmark harness for AI agents.")
console = Console(force_jupyter=False, no_color=True, highlight=False)  # L64 pitfall: Rich + Jupyter + CliRunner


def _version_callback(value: bool):
    if value:
        try:
            v = _pkg_version("agent-bench")
        except PackageNotFoundError:
            v = "0.0.0-dev"
        typer.echo("agent-bench " + v)
        raise typer.Exit()


@app.callback()
def main(
    version: bool = typer.Option(
        False, "--version", callback=_version_callback, is_eager=True, help="Show version and exit."
    ),
):
    # agent-bench: a pluggable benchmark harness for AI agents.
    pass


DEFAULT_SUITE = [
    Task(id="calc-1", category="tool_use", prompt="Compute: 2+2 = 4", env_name="calc", scorer_name="exact_match"),
    Task(id="calc-2", category="tool_use", prompt="Compute: 6*7 = 42", env_name="calc", scorer_name="exact_match"),
    Task(id="file-1", category="file_edit", prompt="Write a result file for this task",
         env_name="file", scorer_name="file_written"),
]


@app.command("run")
def cmd_run(
    k: int = typer.Option(1, "--k", help="Attempts per task (for pass@k)."),
    concurrency: int = typer.Option(4, "--concurrency", "-c", help="Max attempts running at once."),
    sync: bool = typer.Option(False, "--sync", help="Use the sequential runner instead of the async one."),
    flake_rate: float = typer.Option(0.0, "--flake-rate", help="Inject a simulated failure rate into MockAgent."),
    fmt: str = typer.Option("table", "--format", "-f", help="table|json"),
):
    # Run the built-in demo suite and report the pass rate.
    runner = AsyncBenchmarkRunner(DEFAULT_SUITE)
    agent = MockAgent(flake_rate=flake_rate)
    if sync:
        results = runner.run_sync(agent, k=k)
    else:
        results = asyncio.run(runner.run_async(agent, k=k, concurrency=concurrency))
    passed = sum(1 for r in results if r.passed)
    total = len(results)
    if fmt == "json":
        typer.echo(_json.dumps({"passed": passed, "total": total, "k": k, "sync": sync}))
    else:
        table = Table(title="agent-bench run")
        table.add_column("metric"); table.add_column("value")
        table.add_row("passed", str(passed) + "/" + str(total))
        table.add_row("k", str(k))
        table.add_row("concurrency", "n/a (--sync)" if sync else str(concurrency))
        console.print(table)
    if passed < total:
        raise typer.Exit(code=1)


@app.command("list-environments")
def list_environments(fmt: str = typer.Option("table", "--format", "-f")):
    # Show every environment the runner can currently resolve by name.
    load_plugins("environments", verbose=False)
    names = sorted(ENVIRONMENT_REGISTRY)
    if fmt == "json":
        typer.echo(_json.dumps(names))
    else:
        table = Table(title="Registered environments")
        table.add_column("name"); table.add_column("class")
        for n in names:
            table.add_row(n, ENVIRONMENT_REGISTRY[n].__qualname__)
        console.print(table)


@app.command("list-agents")
def list_agents():
    # Show every agent registered by name.
    table = Table(title="Registered agents")
    table.add_column("name")
    for n in sorted(AGENT_REGISTRY):
        table.add_row(n)
    console.print(table)
    if not AGENT_REGISTRY:
        console.print("[no named agents registered yet -- agents are not plugin-discoverable in v0.1.0]")


@app.command("list-scorers")
def list_scorers():
    # Show every scorer the runner can currently resolve by name.
    table = Table(title="Registered scorers")
    table.add_column("name")
    for n in sorted(SCORER_REGISTRY):
        table.add_row(n)
    console.print(table)


@app.command("list-plugins")
def list_plugins(kind: str = typer.Option("environments", help="environments|agents|scorers")):
    # Show installed-but-not-yet-loaded third-party plugins for a given entry-point group.
    found = discover_plugins(kind)
    table = Table(title="Discovered " + repr(kind) + " plugins (installed, not yet loaded)")
    table.add_column("entry point")
    for f in found:
        table.add_row(f)
    console.print(table)
    if not found:
        console.print("[no third-party plugins installed for this group]")


if __name__ == "__main__":
    app()
""")
print("cli.py written -- 5 commands: run, list-environments, list-agents, list-scorers, list-plugins")

## Step 3 -- `pyproject.toml` and an editable install

Same `hatchling` backend and `[project.scripts]` mechanism as L58. Two things that are new for `agent-bench` specifically: the `claude` optional-dependency group (this package's core has zero LLM SDK dependency -- only `ClaudeToolAgent`, unused today, would need `anthropic`) and the `[project.entry-points."agent_bench.environments"]` table, which stays present but empty in `agent-bench`'s own `pyproject.toml` -- it's third-party packages like L66's `agent-bench-shell-plugin` that populate it, never `agent-bench` itself. No `readme` key yet -- that file doesn't exist until L69's OSS Growth lesson, and declaring a `readme` path that doesn't exist would break the build.

In [ ]:
write_file("pyproject.toml", """
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "agent-bench"
version = "0.1.0"
description = "A pluggable benchmark harness for AI agents: Task, Environment, Agent, Scorer, all extensible via plugins."
requires-python = ">=3.10"
license = {text = "MIT"}
authors = [{name = "Gourav Khanijoe"}]
classifiers = [
    "Development Status :: 3 - Alpha",
    "Intended Audience :: Developers",
    "License :: OSI Approved :: MIT License",
    "Programming Language :: Python :: 3.10",
    "Programming Language :: Python :: 3.11",
    "Programming Language :: Python :: 3.12",
]
dependencies = [
    "pydantic>=2.0",
    "typer>=0.9",
    "rich>=13.0",
]

[project.optional-dependencies]
claude = ["anthropic>=0.40"]
dev = ["pytest>=7.0", "build", "twine"]
all = ["agent-bench[claude,dev]"]

[project.urls]
Homepage = "https://github.com/gouravkhanijoe/agent-bench"
Issues = "https://github.com/gouravkhanijoe/agent-bench/issues"

[project.scripts]
agent-bench = "agent_bench.cli:app"

[project.entry-points."agent_bench.environments"]

[tool.hatch.build.targets.wheel]
packages = ["agent_bench"]
""")

r = run(sys.executable, "-m", "pip", "install", "-e", str(PKG_DIR), "-q")
assert r.returncode == 0

# pip installs an editable-mode .pth/import-hook into site-packages, but this NOTEBOOK KERNEL's
# sys.path was already computed at process startup -- site.py won't re-scan for new .pth files
# on its own mid-process. Without this, the very next import of agent_bench would fail with
# ModuleNotFoundError even though the install genuinely succeeded (a real bug hit while building
# this lesson). A fresh `python` process -- a real terminal, unlike this long-lived kernel --
# would never need this; it only matters because we keep one interpreter alive across every cell.
# Note: without an active venv, pip silently prefers the user site-packages dir over
# sysconfig's purelib -- both get scanned here so it works either way.
import site, importlib
site.addsitedir(sysconfig.get_path("purelib"))
site.addsitedir(site.getusersitepackages())
importlib.invalidate_caches()

from importlib.metadata import version as _pkgver
print("agent-bench installed, version:", _pkgver("agent-bench"))

## Step 4 -- smoke-testing the CLI two ways

`typer.testing.CliRunner` (used in every CLI-touching lesson since L58) checks the Python-level app object directly -- fast, but it never proves the console-script entry point actually resolves. The second check is the one that matters more here: after `pip install -e .`, `agent-bench` is a real executable on `$PATH` (or, if the sandbox's script directory isn't on `$PATH`, `python -m agent_bench.cli` -- the `if __name__ == "__main__": app()` guard in `cli.py` makes that fallback work identically). Both paths get exercised below.

In [ ]:
from typer.testing import CliRunner
import agent_bench.cli as abcli
import importlib
importlib.reload(abcli)

cli_runner = CliRunner()

r = cli_runner.invoke(abcli.app, ["--version"])
print("$ agent-bench --version  (CliRunner, exit=" + str(r.exit_code) + ")")
print(r.stdout)
assert r.exit_code == 0 and "agent-bench" in r.stdout

r = cli_runner.invoke(abcli.app, ["run", "--k", "2"])
print("$ agent-bench run --k 2  (CliRunner, exit=" + str(r.exit_code) + ")")
print(r.stdout)
assert r.exit_code == 0

r = cli_runner.invoke(abcli.app, ["list-environments", "--format", "json"])
assert r.exit_code == 0 and "calc" in r.stdout and "file" in r.stdout
print("CliRunner checks passed: --version, run, list-environments --format json")

# Now the real console script -- proves the [project.scripts] entry point actually resolves,
# not just the in-process Typer app object.
candidate_dirs = [sysconfig.get_path("scripts"), os.path.expanduser("~/.local/bin"), "/usr/local/bin"]
for d in candidate_dirs:
    if d and d not in os.environ["PATH"]:
        os.environ["PATH"] = d + ":" + os.environ["PATH"]

agent_bench_bin = shutil.which("agent-bench")
if agent_bench_bin:
    base_cmd = [agent_bench_bin]
    print("Found real console script on PATH:", agent_bench_bin)
else:
    base_cmd = [sys.executable, "-m", "agent_bench.cli"]
    print("Console script not resolvable on this sandbox PATH -- falling back to `python -m agent_bench.cli`"
          " (same code path Typer wires up, exercised via the __main__ guard instead of the entry point).")

_cli_env = dict(os.environ, NO_COLOR="1", TERM="dumb", COLUMNS="200")  # plain, wide, ANSI-free
                                                                        # output -- Typer's --help
                                                                        # renderer otherwise wraps
                                                                        # and color-codes flag names
                                                                        # like --install-completion
                                                                        # mid-string at narrow widths

def cli(*args):
    result = subprocess.run(base_cmd + list(args), capture_output=True, text=True, env=_cli_env)
    print("$ agent-bench " + " ".join(args) + "  (exit=" + str(result.returncode) + ")")
    print(result.stdout)
    return result

res = cli("--version")
assert res.returncode == 0 and "agent-bench" in res.stdout

res = cli("--help")
assert res.returncode == 0 and "list-plugins" in res.stdout

res = cli("run", "--format", "json")
assert res.returncode == 0

res = cli("list-plugins")
assert res.returncode == 0

print("Real subprocess CLI checks passed.")

## Step 5 -- build wheel + sdist, `twine check`

Exactly the two commands CI runs before any PyPI publish (same as L58 and L64's `paper-distiller`): `python -m build` produces a `.whl` and a `.tar.gz` under `dist/`, and `twine check` validates the packaging metadata (long description rendering, required fields) without uploading anything.

In [ ]:
r = run(sys.executable, "-m", "build", str(PKG_DIR))
assert r.returncode == 0, "build failed"

dist_dir = PKG_DIR / "dist"
dist_files = sorted(str(p) for p in dist_dir.glob("*"))
print("Built:", dist_files)
assert any(f.endswith(".whl") for f in dist_files)
assert any(f.endswith(".tar.gz") for f in dist_files)

r = run(sys.executable, "-m", "twine", "check", *dist_files)
assert r.returncode == 0, "twine check failed"
print("twine check: PASSED")

## Step 6 -- fresh-install verification

`pip install -e .` proves the code runs from its checked-out location. It does *not* prove the built wheel is correct -- hatchling's package-discovery config (`[tool.hatch.build.targets.wheel] packages = ["agent_bench"]`) or an accidentally-excluded file can produce a wheel that installs fine but is missing modules, and an editable install would never catch that because it is not importing from the wheel at all. So: uninstall the editable version, install the actual `.whl` from `dist/`, and re-run the same version check.

In [ ]:
run(sys.executable, "-m", "pip", "uninstall", "-y", "-q", "agent-bench")

wheel_path = next(p for p in dist_dir.glob("*.whl"))
r = run(sys.executable, "-m", "pip", "install", "-q", str(wheel_path))
assert r.returncode == 0

# Reload metadata cache and re-check via the real console script -- this time it is running
# code unpacked from the wheel, not the editable source tree.
import importlib.metadata
importlib.reload(importlib.metadata)
print("agent-bench version (from wheel):", importlib.metadata.version("agent-bench"))

res = cli("--version")
assert res.returncode == 0 and "agent-bench" in res.stdout
res = cli("list-scorers")
assert res.returncode == 0 and "exact_match" in res.stdout
print("Wheel-install verification passed: the built artifact, not just the source tree, works end to end.")

## Step 7 -- the PyPI publishing runbook

Same honest boundary L64 drew for `paper-distiller`: everything up to here runs inside this sandbox with zero external identity. Publishing to PyPI requires Gourav's own GitHub and PyPI accounts, so this step is a runbook, not a command that executes here.

In [ ]:
runbook = """
agent-bench v0.1.0 -- publish runbook (manual, requires your GitHub + PyPI identity)

  1. git init && git add -A && git commit -m "agent-bench v0.1.0"
  2. gh repo create agent-bench --public --source=. --push
  3. TestPyPI dry run first:
       python -m twine upload --repository testpypi dist/*
       pip install --index-url https://test.pypi.org/simple/ agent-bench
  4. Register a PyPI Trusted Publisher (OIDC, no API token to leak or rotate):
       pypi.org -> your project -> Publishing -> Add a new publisher
       owner=<your-gh-username>  repo=agent-bench  workflow=release.yml  environment=pypi
  5. Watch ci.yml go green on the push from step 1 BEFORE tagging -- do not tag a red build.
  6. git tag v0.1.0 && git push origin v0.1.0   # triggers release.yml (see Step 8 below)
  7. Verify from a clean environment:  pip install agent-bench && agent-bench --version
  8. File the first good-first-issue (same practice as L64): agents/scorers are not yet
     plugin-discoverable via entry_points (see Step 2's flagged asymmetry) -- good first issue.
"""
print(runbook)

## Step 8 -- CI + release workflows

`ci.yml` runs on every push/PR: install `agent-bench[dev]`, run the CliRunner smoke tests as real pytest, build, `twine check`. `release.yml` triggers only on a pushed `v*` tag and publishes via OIDC Trusted Publishing (`id-token: write`, no `PYPI_API_TOKEN` secret to manage) -- identical mechanism to L58 and L64.

In [ ]:
write_file(".github/workflows/ci.yml", """
name: CI
on:
  push:
  pull_request:
jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.10", "3.11", "3.12"]
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      - run: pip install -e ".[dev]"
      - run: pytest -q
      - run: python -m build
      - run: twine check dist/*
""")

write_file(".github/workflows/release.yml", """
name: Release
on:
  push:
    tags:
      - "v*"
permissions:
  id-token: write
jobs:
  publish:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - run: pip install build
      - run: python -m build
      - uses: pypa/gh-action-pypi-publish@release/v1
""")
print("ci.yml + release.yml written")

## Step 9 -- shell completion

Typer adds `--install-completion` and `--show-completion` to every app for free (unless `add_completion=False` is passed to `typer.Typer()`, which `cli.py` does not). This is documentation, not an executed install -- writing to a user's shell rc file isn't something to do from inside a lesson.

In [ ]:
res = cli("--help")
assert "--install-completion" in res.stdout and "--show-completion" in res.stdout
print("Confirmed: --install-completion / --show-completion are present, free from Typer.")
print("Real usage (run this yourself in a terminal, not here): agent-bench --install-completion")

## Pitfalls

| # | Pitfall | Why it bites |
|---|---|---|
| 1 | Hardcoding the version string in `cli.py` instead of reading it via `importlib.metadata.version()` | Drifts from `pyproject.toml` the first time someone bumps one and forgets the other |
| 2 | Forgetting `is_eager=True` on `--version` | Typer evaluates options in declaration order by default; a required positional arg placed before an eager-less `--version` would demand a value before the version check ever runs |
| 3 | Mixing a Rich table into `--format json` output | Machine consumers (`jq`, CI) need stdout to be *only* the JSON payload -- any extra `console.print()` on that path corrupts it |
| 4 | Trusting a lesson's "condensed from L61/L65" recreation as complete | Exactly today's opening bug: L67 silently dropped L66's plugin discovery because that lesson didn't need it -- always diff a condensed recreation against the last full version before building on top of it |
| 5 | `pip install -e .` passing while the built wheel is broken | Editable installs import from the source tree directly; `[tool.hatch.build.targets.wheel] packages=[...]` misconfiguration only shows up once you actually install the `.whl` (Step 6 exists specifically to catch this) |
| 6 | `twine check` treated as "the package works" | It validates packaging *metadata* only (README rendering, required fields) -- it does not import your code or run anything |
| 7 | Console-script entry point silently missing from `$PATH` | `pip install` (no `--user`, no active venv) can put scripts in a `bin/` directory that isn't on `$PATH` in a restricted sandbox -- real thing hit while building this lesson; `python -m agent_bench.cli` is the portable fallback, which is why `cli.py` ends with `if __name__ == "__main__": app()` |
| 8 | Tagging and pushing a release before CI is green | Same L58/L64 mistake, still possible even with OIDC configured correctly |
| 9 | Calling `asyncio.run()` inside a notebook cell the same way the CLI's `run` command does | `asyncio.run()` requires no event loop already running; a terminal process has none, but a Jupyter/Colab kernel already has one -- L61-L67 handled this with `nest_asyncio.apply()`, and that guard has to travel with any notebook demo of the CLI's async path, not just the `.py` module |
| 10 | Declaring `readme = "README.md"` in `pyproject.toml` before the file exists | Breaks `python -m build` outright -- deliberately deferred to L69's OSS Growth lesson rather than stubbing a placeholder file today |
| 11 | Importing a package right after `pip install -e .` in the SAME long-lived process | A live interpreter's `sys.path`/import machinery was already set up at startup; new `.pth`/editable-install hooks written to site-packages mid-process need `site.addsitedir(...)` + `importlib.invalidate_caches()` before the next `import` will find them -- a real `ModuleNotFoundError` hit while building this lesson, and irrelevant in a fresh terminal `pip install` where no code has imported anything yet |


In [ ]:
checks = {
    "core.py exists": (PKG_DIR / "agent_bench" / "core.py").exists(),
    "registry.py exists": (PKG_DIR / "agent_bench" / "registry.py").exists(),
    "environments.py exists": (PKG_DIR / "agent_bench" / "environments.py").exists(),
    "runner_async.py exists": (PKG_DIR / "agent_bench" / "runner_async.py").exists(),
    "cli.py exists": (PKG_DIR / "agent_bench" / "cli.py").exists(),
    "pyproject.toml exists": (PKG_DIR / "pyproject.toml").exists(),
    "ci.yml exists": (PKG_DIR / ".github" / "workflows" / "ci.yml").exists(),
    "release.yml exists": (PKG_DIR / ".github" / "workflows" / "release.yml").exists(),
    "wheel built": any(dist_dir.glob("*.whl")),
    "sdist built": any(dist_dir.glob("*.tar.gz")),
    "agent-bench importable": "agent_bench.cli" in sys.modules,
    "discover_plugins merged back into registry.py": hasattr(abcli, "discover_plugins"),
    "load_plugins merged back into registry.py": hasattr(abcli, "load_plugins"),
    "ENVIRONMENT_REGISTRY has calc+file": {"calc", "file"} <= set(abcli.ENVIRONMENT_REGISTRY),
    "SCORER_REGISTRY has exact_match+file_written": {"exact_match", "file_written"} <= set(abcli.SCORER_REGISTRY),
}

for name, ok in checks.items():
    print(("PASS" if ok else "FAIL") + " -- " + name)
assert all(checks.values()), "one or more verification checks failed"
print()
print(str(len(checks)) + "/" + str(len(checks)) + " checks passed.")

## Summary

| Concept | What it is |
|---|---|
| `importlib.metadata.version()` in `--version` | Reads the truth from the installed distribution instead of a string that can drift from `pyproject.toml` |
| `is_eager=True` | Forces an option's callback to run before other argument parsing/validation, required for `--version`/`--help`-style short-circuit flags |
| `--format table\|json` | The difference between a CLI a human reads and a CLI a script or CI job can consume |
| `[project.scripts]` | The `pyproject.toml` table that turns a Python function into a real OS-level executable on install |
| Editable vs. wheel install | Editable proves the source tree works; only installing the actual built `.whl` proves the *packaging config* is correct |
| `twine check` | Validates packaging metadata only -- not a substitute for actually running the installed package |
| OIDC Trusted Publishing | PyPI releases with zero long-lived API tokens to leak or rotate, gated on a GitHub Actions workflow identity |
| Reconciling condensed recreations | This lesson's own opening move: catching that L67 silently dropped L66's plugin discovery, and merging it back deliberately |

**Homework:**
1. Actually run the Step 7 runbook end to end: real GitHub repo, real TestPyPI upload, verify `pip install` from TestPyPI.
2. Register `MockAgent` under `AGENT_REGISTRY` via `register_agent("mock")` and update `list-agents` to show it -- resolves the asymmetry flagged in Step 2.
3. Add a `pytest` test file (`tests/test_cli.py`) exercising every command via `CliRunner`, and wire it into `ci.yml`'s `pytest -q` step for real (today `ci.yml` is written to disk but not run against these exact tests).
4. Build a second real environment or scorer plugin package (like L66's `ShellEnv`) and confirm `agent-bench list-plugins` discovers it after `pip install -e`.
5. Add a `agent-bench run --json-lines` streaming mode for long suites, following L57's JSONL batch-output pattern.

**L69 preview:** OSS Growth for `agent-bench` -- README, badges, CONTRIBUTING funnel, issue templates, following L60's playbook for `paper-distiller`. The `readme = "README.md"` line deferred in Step 3 gets filled in then.